# 原生共有 regulon AUCell Pearson（外部大文件模式）

此副本保留源 notebook 的全部历史 cell 输出；仅适配路径，本次整理不重新读取全量 AUCell/H5AD。运行需 `manifests/workflow_manifest.json` 登记的六份外部推断结果 H5AD。包内已提供 898 行 Pearson 结果与配对质量表，绘图不依赖本 notebook 的重新执行。

历史来源：`14_shared_regulon_Pearson.ipynb`。两组各用自身最终 regulon 的原生评分，在字符串 bin ID 上对齐全部 117,867 个 bin；缺失模块不填零相关。

本次只为外部文件按字节计算 SHA256，没有加载这些外部矩阵重新进行评分、相关分析或表达汇总。


In [1]:
from pathlib import Path
import json
import h5py
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from IPython.display import display

ROOT = Path.cwd()
BASE = ROOT
RESULTS = BASE / 'data/analysis_recomputed'
manifest = json.loads((BASE / 'manifests/workflow_manifest.json').read_text())
source_specs = [manifest['full'], *manifest['runs']]
sources = pd.DataFrame([{'run': spec['run'], 'native_auc_h5ad': spec['result'],
                         'regulons_json': spec['regulons'], 'inference_notebook': spec['notebook']}
                        for spec in source_specs])
sources.to_csv(RESULTS / 'native_shared_regulon_pearson_sources.tsv', sep='\t', index=False)
sources

,run,native_auc_h5ad,regulons_json,inference_notebook
0,full,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...
1,rep01,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...
2,rep02,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...
3,rep03,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...
4,rep04,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...
5,rep05,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...


In [2]:
def read_native_auc(path):
    with h5py.File(path, 'r') as handle:
        group = handle['obsm/auc_mtx']
        ids = group[group.attrs['_index']].asstr()[:]
        return pd.DataFrame({name: group[name][:] for name in group.attrs['column-order']},
                            index=pd.Index(ids, name='bin_id'))

full_auc = read_native_auc(manifest['full']['result'])
full_regulons = json.loads(Path(manifest['full']['regulons']).read_text())
records = []
quality_rows = []
for spec in manifest['runs']:
    native_sample = read_native_auc(spec['result'])
    sample_regulons = json.loads(Path(spec['regulons']).read_text())
    shared = sorted(full_regulons.keys() & sample_regulons.keys())
    sample_auc = native_sample.loc[full_auc.index, shared]
    for name in shared:
        x = full_auc[name].to_numpy()
        y = sample_auc[name].to_numpy()
        records.append({'run': spec['run'], 'seed': spec['seed'], 'regulon': name,
                        'n_bins': len(x), 'pearson': float(pearsonr(x, y)[0]),
                        'n_targets_full': len(full_regulons[name]),
                        'n_targets_sample': len(sample_regulons[name])})
    quality_rows.append({'run': spec['run'], 'n_bins_full': len(full_auc),
        'n_bins_sample': len(native_sample), 'n_shared_regulons': len(shared),
        'full_bin_ids_unique': full_auc.index.is_unique,
        'sample_bin_ids_unique': native_sample.index.is_unique,
        'same_bin_order': full_auc.index.equals(native_sample.index),
        'aligned_bin_order': full_auc.index.equals(sample_auc.index),
        'full_auc_matches_final_regulons': set(full_auc.columns) == set(full_regulons),
        'sample_auc_matches_final_regulons': set(native_sample.columns) == set(sample_regulons),
        'nonfinite_full_scores': int((~np.isfinite(full_auc[shared].to_numpy())).sum()),
        'nonfinite_sample_scores': int((~np.isfinite(sample_auc.to_numpy())).sum())})

metrics = pd.DataFrame(records)
quality = pd.DataFrame(quality_rows)
metrics.to_csv(RESULTS / 'native_shared_regulon_pearson.tsv', sep='\t', index=False)
quality.to_csv(RESULTS / 'native_shared_regulon_pearson_quality.tsv', sep='\t', index=False)
display(metrics.head())
display(quality)

,run,seed,regulon,n_bins,pearson,n_targets_full,n_targets_sample
0,rep01,20261016,ALX3(+),117867,0.201054,9,5
1,rep01,20261016,ARNT2(+),117867,0.857029,904,490
2,rep01,20261016,ATF1(+),117867,0.880466,203,56
3,rep01,20261016,ATF2(+),117867,0.331079,126,42
4,rep01,20261016,ATF4(+),117867,0.889262,625,360


,run,n_bins_full,n_bins_sample,n_shared_regulons,full_bin_ids_unique,sample_bin_ids_unique,same_bin_order,aligned_bin_order,full_auc_matches_final_regulons,sample_auc_matches_final_regulons,nonfinite_full_scores,nonfinite_sample_scores
0,rep01,117867,117867,183,True,True,True,True,True,True,0,0
1,rep02,117867,117867,181,True,True,True,True,True,True,0,0
2,rep03,117867,117867,181,True,True,True,True,True,True,0,0
3,rep04,117867,117867,178,True,True,True,True,True,True,0,0
4,rep05,117867,117867,175,True,True,True,True,True,True,0,0


In [3]:
summary = metrics.groupby(['run', 'seed'], sort=False).agg(
    n_shared=('regulon', 'size'), n_valid=('pearson', 'count'),
    min_paired_bins=('n_bins', 'min'), mean=('pearson', 'mean'), sd=('pearson', 'std'),
    median=('pearson', 'median'), q25=('pearson', lambda x: x.quantile(0.25)),
    q75=('pearson', lambda x: x.quantile(0.75)), minimum=('pearson', 'min'), maximum=('pearson', 'max')
).reset_index()
summary.to_csv(RESULTS / 'native_shared_regulon_pearson_summary.tsv', sep='\t', index=False)
display(summary)
display(metrics['pearson'].describe().to_frame('Pearson across shared regulon pairs'))

,run,seed,n_shared,n_valid,min_paired_bins,mean,sd,median,q25,q75,minimum,maximum
0,rep01,20261016,183,183,117867,0.750525,0.237628,0.838439,0.739275,0.903099,-0.198366,0.957953
1,rep02,20261017,181,181,117867,0.768526,0.213743,0.839613,0.738871,0.902853,-0.285936,0.953405
2,rep03,20261018,181,181,117867,0.765021,0.220007,0.839606,0.743295,0.906367,-0.104398,0.957422
3,rep04,20261019,178,178,117867,0.775626,0.193609,0.841161,0.752485,0.900752,-0.090019,0.958912
4,rep05,20261020,175,175,117867,0.768988,0.212666,0.831836,0.737096,0.903441,-0.129458,0.957591


,Pearson across shared regulon pairs
count,898.000000
mean,0.765648
std,0.215819
min,-0.285936
25%,0.741136
50%,0.839346
75%,0.903611
max,0.958912
